# Colab GPU 연동 AI · ML · DL 실습

Google Colab에서 런타임 유형을 T4 GPU로 바꾸고 모두 실행하세요. 코드가 CUDA를 자동 감지하며, GPU를 쓸 수 있으면 모든 모델과 텐서를 GPU에서 실행합니다. 외부 데이터 다운로드가 필요 없는 합성 데이터 실습입니다.

실습 5개: GPU 텐서 시각화 · 선형 회귀 · MLP 비선형 분류 · CNN 이미지 분류 · 혼동 행렬 평가.

로컬 미리보기 PNG는 실행 환경을 명확히 표시합니다. Colab에서 다시 실행하면 그래프 제목에 실제 GPU 이름이 기록됩니다.

In [ ]:
import os, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    device = torch.device("cuda")
    RUN_LABEL = ("COLAB GPU · " if IN_COLAB else "CUDA GPU · ") + torch.cuda.get_device_name(0)
else:
    device = torch.device("cpu")
    RUN_LABEL = "LOCAL CPU PREVIEW" if not IN_COLAB else "COLAB CPU (GPU not enabled)"
OUT = Path("/content/mai_sma_results") if IN_COLAB else (Path.cwd() / "screenshots")
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"savefig.dpi":160,"axes.spines.top":False,"axes.spines.right":False,"font.size":11})
print(f"Runtime: {RUN_LABEL}")
print(f"PyTorch {torch.__version__} | device={device} | output={OUT}")

## 1. GPU 연결과 텐서 연산
장치에서 대규모 텐서를 계산하고 값의 분포를 그래프로 확인합니다.

In [ ]:
x = torch.randn(250_000, device=device)
y = torch.sin(x) + 0.15 * torch.randn_like(x)
print(f"Device: {device} | samples: {x.numel():,}")
print(f"mean={y.mean().item():.4f} | std={y.std().item():.4f}")
fig, ax = plt.subplots(figsize=(9,5))
ax.hist(y.detach().cpu().numpy(), bins=55, color="#4778d0", alpha=.88, edgecolor="white")
ax.set(title=f"01 · Tensor distribution | {RUN_LABEL}", xlabel="Tensor value", ylabel="Frequency")
ax.grid(axis="y", alpha=.22)
fig.tight_layout(); fig.savefig(OUT/"01_gpu_tensor_distribution.png"); plt.show(); plt.close(fig)

## 2. 머신러닝 선형 회귀
기울기와 절편이 정해진 합성 데이터에서 y = ax + b를 학습합니다.

In [ ]:
g = torch.Generator().manual_seed(SEED)
xr = torch.linspace(-3,3,320).unsqueeze(1)
yr = 2.4*xr - .7 + .55*torch.randn(xr.shape,generator=g)
perm = torch.randperm(len(xr),generator=g); train_idx,test_idx = perm[:256],perm[256:]
xtr,ytr = xr[train_idx].to(device),yr[train_idx].to(device)
xte,yte = xr[test_idx].to(device),yr[test_idx].to(device)
reg = nn.Linear(1,1).to(device); opt = optim.Adam(reg.parameters(),lr=.08); loss_fn=nn.MSELoss(); reg_losses=[]
for epoch in range(240):
    opt.zero_grad(); pred=reg(xtr); loss=loss_fn(pred,ytr); loss.backward(); opt.step(); reg_losses.append(loss.item())
with torch.no_grad():
    test_mse=loss_fn(reg(xte),yte).item(); slope=reg.weight.item(); intercept=reg.bias.item()
print(f"Learned equation: y={slope:.3f}x {intercept:+.3f} | Test MSE={test_mse:.4f}")
fig,ax=plt.subplots(1,2,figsize=(12,4.7))
ax[0].scatter(xr.numpy(),yr.numpy(),s=14,alpha=.42,color="#7c91ad",label="observations")
line_x=torch.linspace(-3.2,3.2,100).unsqueeze(1).to(device)
with torch.no_grad(): line_y=reg(line_x).cpu().numpy()
ax[0].plot(line_x.cpu().numpy(),line_y,color="#e47745",lw=2.6,label="learned fit")
ax[0].set(title=f"Linear regression | {RUN_LABEL}",xlabel="x",ylabel="y"); ax[0].legend(frameon=False); ax[0].grid(alpha=.2)
ax[1].plot(reg_losses,color="#4778d0",lw=2); ax[1].set(title="Training loss (MSE)",xlabel="Epoch",ylabel="Loss"); ax[1].grid(alpha=.2)
fig.tight_layout(); fig.savefig(OUT/"02_ml_linear_regression.png"); plt.show(); plt.close(fig)

## 3. 머신러닝 분류와 딥러닝 MLP
두 곡선 클래스를 다층 퍼셉트론으로 분류하고 결정 경계를 확인합니다.

In [ ]:
def make_moons(n=900,noise=.12,seed=SEED):
    r=np.random.default_rng(seed); n0=n//2; t=r.uniform(0,np.pi,n0)
    a=np.c_[np.cos(t),np.sin(t)]; b=np.c_[1-np.cos(t),.5-np.sin(t)]
    X=np.vstack([a,b])+r.normal(0,noise,(2*n0,2))
    return X.astype("float32"),np.r_[np.zeros(n0),np.ones(n0)].astype("int64")
Xm,ym=make_moons(); order=np.random.default_rng(SEED).permutation(len(ym)); cut=int(.8*len(ym))
Xtr=torch.tensor(Xm[order[:cut]],device=device); ytr=torch.tensor(ym[order[:cut]],device=device).float().unsqueeze(1)
Xte=torch.tensor(Xm[order[cut:]],device=device); yte=torch.tensor(ym[order[cut:]],device=device).long()
mlp=nn.Sequential(nn.Linear(2,32),nn.ReLU(),nn.Linear(32,32),nn.ReLU(),nn.Linear(32,1)).to(device)
opt=optim.Adam(mlp.parameters(),lr=.012); bce=nn.BCEWithLogitsLoss(); mlp_losses=[]
for epoch in range(220):
    opt.zero_grad(); logits=mlp(Xtr); loss=bce(logits,ytr); loss.backward(); opt.step(); mlp_losses.append(loss.item())
with torch.no_grad():
    prob=mlp(Xte).sigmoid().squeeze(); mlp_acc=((prob>=.5).long()==yte).float().mean().item()
xx,yy=np.meshgrid(np.linspace(Xm[:,0].min()-.35,Xm[:,0].max()+.35,220),np.linspace(Xm[:,1].min()-.35,Xm[:,1].max()+.35,220))
with torch.no_grad(): zz=mlp(torch.tensor(np.c_[xx.ravel(),yy.ravel()],dtype=torch.float32,device=device)).sigmoid().cpu().numpy().reshape(xx.shape)
print(f"MLP test accuracy: {mlp_acc*100:.1f}%")
fig,ax=plt.subplots(1,2,figsize=(12,4.8))
ax[0].contourf(xx,yy,zz,levels=[0,.5,1],colors=["#dce8fa","#fbe4d9"],alpha=.88); ax[0].contour(xx,yy,zz,levels=[.5],colors=["#263f60"],linewidths=1.3)
ax[0].scatter(Xm[:,0],Xm[:,1],c=ym,cmap=matplotlib.colors.ListedColormap(["#4778d0","#e47745"]),s=14,edgecolors="white",linewidths=.25)
ax[0].set(title=f"MLP decision boundary · {mlp_acc*100:.1f}% | {RUN_LABEL}",xlabel="Feature 1",ylabel="Feature 2")
ax[1].plot(mlp_losses,color="#8a61cc",lw=2); ax[1].set(title="MLP training loss",xlabel="Epoch",ylabel="Binary cross-entropy"); ax[1].grid(alpha=.2)
fig.tight_layout(); fig.savefig(OUT/"03_ml_mlp_classification.png"); plt.show(); plt.close(fig)

## 4. 딥러닝 CNN 이미지 분류
세로선·가로선·대각선 패턴을 이미지로 생성하고 합성곱 신경망(CNN)으로 분류합니다.

In [ ]:
def make_patterns(n,seed):
    r=np.random.default_rng(seed); imgs=np.zeros((n,1,16,16),dtype="float32"); labels=np.arange(n)%3; r.shuffle(labels)
    for i,c in enumerate(labels):
        shift=int(r.integers(-1,2))
        if c==0:
            for col in (4+shift,11+shift): imgs[i,0,:,max(0,col):min(16,col+2)]=1
        elif c==1:
            for row in (4+shift,11+shift): imgs[i,0,max(0,row):min(16,row+2),:]=1
        else:
            for k in range(3,13):
                for d in range(2):
                    rr=k+shift; cc=k+d
                    if 0<=rr<16 and 0<=cc<16: imgs[i,0,rr,cc]=1
                    rr=k+shift; cc=15-k+d
                    if 0<=rr<16 and 0<=cc<16: imgs[i,0,rr,cc]=1
    imgs=np.clip(imgs+r.normal(0,.18,imgs.shape),0,1).astype("float32")
    return torch.tensor(imgs),torch.tensor(labels,dtype=torch.long)
Ximg,yimg=make_patterns(1050,SEED); Xit,yit=make_patterns(300,SEED+1)
Ximg=Ximg.to(device); yimg=yimg.to(device); Xit=Xit.to(device); yit=yit.to(device)
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(1,12,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(12,24,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Flatten(),nn.Linear(24*4*4,48),nn.ReLU(),nn.Linear(48,3))
    def forward(self,x): return self.net(x)
cnn=TinyCNN().to(device); opt=optim.Adam(cnn.parameters(),lr=.003); ce=nn.CrossEntropyLoss(); cnn_losses=[]; cnn_accs=[]
for epoch in range(28):
    cnn.train(); order=torch.randperm(len(Ximg),device=device); running=0
    for ids in order.split(96):
        opt.zero_grad(); logits=cnn(Ximg[ids]); loss=ce(logits,yimg[ids]); loss.backward(); opt.step(); running+=loss.item()*len(ids)
    cnn_losses.append(running/len(Ximg)); cnn.eval()
    with torch.no_grad(): cnn_accs.append((cnn(Xit).argmax(1)==yit).float().mean().item())
with torch.no_grad(): preds=cnn(Xit).argmax(1)
cnn_acc=(preds==yit).float().mean().item()
print(f"CNN test accuracy: {cnn_acc*100:.1f}% | classes: vertical, horizontal, diagonal")
fig,ax=plt.subplots(1,2,figsize=(12,4.8))
examples=[]
for c in range(3): examples.append(Xit[int(torch.where(yit==c)[0][0]),0].cpu().numpy())
ax[0].imshow(np.concatenate(examples,axis=1),cmap="Blues",vmin=0,vmax=1,extent=[0,3,0,1],aspect="auto")
ax[0].set_xlim(0,3); ax[0].set_xticks([.5,1.5,2.5],["Vertical","Horizontal","Diagonal"]); ax[0].set_yticks([]); ax[0].set_title("Synthetic image examples")
ax[1].plot(cnn_losses,label="train loss",color="#4778d0",lw=2); axr=ax[1].twinx(); axr.plot(cnn_accs,label="test accuracy",color="#e47745",lw=2)
ax[1].set(title=f"CNN learning curves | {RUN_LABEL}",xlabel="Epoch",ylabel="Cross-entropy"); axr.set_ylabel("Test accuracy"); axr.set_ylim(0,1.04); ax[1].grid(alpha=.2)
lines,labels=ax[1].get_legend_handles_labels(); lines2,labels2=axr.get_legend_handles_labels(); ax[1].legend(lines+lines2,labels+labels2,frameon=False,loc="center right")
fig.tight_layout(); fig.savefig(OUT/"04_dl_cnn_image_classification.png"); plt.show(); plt.close(fig)

## 5. 모델 평가와 혼동 행렬
테스트 이미지 예측을 혼동 행렬과 클래스별 정밀도·재현율·F1 점수로 요약합니다.

In [ ]:
cm=torch.zeros((3,3),dtype=torch.int64)
for t,p in zip(yit.cpu(),preds.cpu()): cm[t,p]+=1
cmn=cm.numpy(); support=cmn.sum(axis=1); tp=np.diag(cmn); precision=tp/np.maximum(cmn.sum(axis=0),1); recall=tp/np.maximum(support,1); f1=2*precision*recall/np.maximum(precision+recall,1e-12); overall=tp.sum()/cmn.sum()
print(f"Overall test accuracy: {overall*100:.2f}%")
for name,p,r,f in zip(["Vertical","Horizontal","Diagonal"],precision,recall,f1): print(f"{name:10s} precision={p:.3f} recall={r:.3f} F1={f:.3f}")
fig,ax=plt.subplots(1,2,figsize=(12,4.8))
im=ax[0].imshow(cmn,cmap="Blues"); ax[0].set(title=f"Confusion matrix · {overall*100:.1f}% | {RUN_LABEL}",xlabel="Predicted class",ylabel="Actual class",xticks=range(3),yticks=range(3))
ax[0].set_xticklabels(["Vertical","Horizontal","Diagonal"],rotation=20); ax[0].set_yticklabels(["Vertical","Horizontal","Diagonal"])
for i in range(3):
    for j in range(3): ax[0].text(j,i,str(cmn[i,j]),ha="center",va="center",color="white" if cmn[i,j]>cmn.max()*.55 else "#18334f",fontweight="bold")
fig.colorbar(im,ax=ax[0],fraction=.046,pad=.04)
xxi=np.arange(3); w=.24
ax[1].bar(xxi-w,precision,w,label="Precision",color="#4778d0"); ax[1].bar(xxi,recall,w,label="Recall",color="#e47745"); ax[1].bar(xxi+w,f1,w,label="F1",color="#57a879")
ax[1].set(xticks=xxi,xticklabels=["Vertical","Horizontal","Diagonal"],ylim=(0,1.08),ylabel="Score",title="Per-class evaluation"); ax[1].legend(frameon=False); ax[1].grid(axis="y",alpha=.2)
fig.tight_layout(); fig.savefig(OUT/"05_dl_confusion_matrix_metrics.png"); plt.show(); plt.close(fig)
zip_path=OUT.parent/"Colab_GPU_AI_ML_DL_결과_5장.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob("*.png")): z.write(p,p.name)
print(f"5 images saved to: {OUT}")
print(f"ZIP: {zip_path}")

## Colab에서 결과 이미지 받기
마지막 셀을 실행하면 5개 그래프가 PNG로 저장되고 ZIP 파일이 만들어집니다. Colab의 파일 패널에서 /content/Colab_GPU_AI_ML_DL_결과_5장.zip을 다운로드하세요. 런타임을 T4 GPU로 설정하면 각 그래프 제목에 사용된 GPU 이름이 표시됩니다.